In [24]:
%uv pip install torch triton

Using Python 3.12.6 environment at: /usr/local
Audited 2 packages in 19ms
Note: you may need to restart the kernel to use updated packages.


In [25]:
import torch

import triton
import triton.language as tl

DEVICE = triton.runtime.driver.active.get_active_torch_device()

In [26]:
@triton.jit
def softmax_k(input_ptr, output_ptr, n_cols, BLOCK_SIZE:tl.constexpr):

    row_idx = tl.program_id(0)
    col_offs = tl.arange(0, BLOCK_SIZE)
    offs = row_idx * n_cols + col_offs
    mask = col_offs < n_cols

    row = tl.load(
        input_ptr + offs,
        mask=mask,
        other=-float("inf")
    )

    row_max = tl.max(row, axis=0)
    shifted_row = row - row_max
    
    numerator = tl.exp(shifted_row)
    denominator = tl.sum(numerator, axis=0)
    
    softmax_output = numerator / denominator
    
    tl.store(
        output_ptr+offs,
        softmax_output,
        mask=mask
    )

In [30]:
def triton_softmax(x):
    assert x.is_cuda
    assert x.dtype == torch.float32
    assert x.ndim == 2
    assert x.is_contiguous()
    assert x.shape[1] > 0

    n_rows, n_cols = x.shape

    output = torch.empty_like(x)

    block_size = triton.next_power_of_2(n_cols)

    grid = (n_rows,)

    softmax_k[grid](
        x, output,
        n_cols,
        BLOCK_SIZE=block_size
    )

    return output

In [28]:
x = torch.tensor(
    [
        [1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0],
    ],
    device=DEVICE,
)

output = triton_softmax(x)
expected = torch.softmax(x, dim=1)

print(output)
print(expected)
print("Row sums:", output.sum(dim=1))

torch.testing.assert_close(output, expected, rtol=1e-5, atol=1e-6)

tensor([[0.0900, 0.2447, 0.6652],
        [0.0900, 0.2447, 0.6652]], device='cuda:0')
tensor([[0.0900, 0.2447, 0.6652],
        [0.0900, 0.2447, 0.6652]], device='cuda:0')
Row sums: tensor([1.0000, 1.0000], device='cuda:0')


In [31]:
widths = [
    1,
    3,
    31,
    128,
    256,
    512,
    1000,
    1024,
    2048,
    4096,
    8192,
]

for n_cols in widths:
    x = torch.randn(
        (17, n_cols),
        device=DEVICE,
        dtype=torch.float32,
    ) * 10

    actual = triton_softmax(x)
    expected = torch.softmax(x, dim=1)

    torch.testing.assert_close(
        actual,
        expected,
        rtol=1e-4,
        atol=1e-6,
    )

    print(
        f"width={n_cols}, "
        f"block={triton.next_power_of_2(n_cols)}: passed"
    )

width=1, block=1: passed
width=3, block=4: passed
width=31, block=32: passed
width=128, block=128: passed
width=256, block=256: passed
width=512, block=512: passed
width=1000, block=1024: passed
width=1024, block=1024: passed
width=2048, block=2048: passed
width=4096, block=4096: passed
width=8192, block=8192: passed
